# Lab 03 – Receiver

The receiving side of the lab. `sniff()` blocks the kernel until it is stopped, so the receiver runs in this second notebook, in another tab next to `lab_03.ipynb` or on the colleague's machine. Imports and constants are prepared; the section numbers follow `lab_03.ipynb` and the [README](../README.md).

Set `IFACE` to the interface the packets arrive on and keep `DST_PORT` and `ENCODING` equal to the sender's constants. Interrupt a running `sniff()` with *Kernel → Interrupt Kernel* (or the stop button).

## Imports

In [ ]:
import socket
import sys
from pathlib import Path

import numpy as np
import panel as pn
import scapy.all as sa
from IPython.display import Audio, display

from lib.capture import (
    decode_rtp,
    rtp_stream,
    skip_loopback_copies,
    stream_card,
    stream_stats,
    stream_view,
)
from lib.core import SAMPLE_RATE

pn.extension("tabulator")

## Constants

In [ ]:
IFACE = "lo"  # interface to sniff on; the laboratory interface for two machines
DST_PORT = 54321  # UDP port the sender sends to (DST_PORT in lab_03.ipynb)
ENCODING = "alaw"  # G.711 variant of the sender (ENCODING in lab_03.ipynb)

CAPTURE_SECONDS = 30  # length of an RTP capture in 2.2 [s]
TRIAL_SECONDS = 60  # length of the sender's test stream in Step 3 [s]
TRIAL_PACKETS = TRIAL_SECONDS * 50  # packets of the test stream at 20 ms

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

## Privileges

Sniffing requires the `CAP_NET_RAW` capability on the interpreter (README, Preparation). The cell prints the command for this kernel's interpreter if it is missing; restart the kernel after running it.

In [ ]:
interpreter = Path(sys.executable).resolve()
try:
    socket.socket(socket.AF_PACKET, socket.SOCK_RAW).close()
    print(f"Raw sockets are available to {interpreter}.")
except PermissionError:
    print(
        "Raw sockets are not available. Run this in a terminal, then restart the kernel:"
    )
    print(f"\n    sudo setcap cap_net_raw=eip {interpreter}\n")
    print("If setcap fails on a network file system, see README, Preparation.")

## Step 2 – Build a traffic generator

### 2.1 Text exchange

Prints the payload of every ICMP echo request. Start it, then run the sender cell in `lab_03.ipynb`.

In [ ]:
def print_payload(pkt):
    if pkt["ICMP"].type == 8:
        print(pkt["Raw"].load.decode(errors="replace"), end="")


# On the loopback interface, Scapy sees each packet twice (leaving and arriving);
# skip_loopback_copies() keeps the first copy only.
sa.sniff(
    iface=IFACE,
    prn=print_payload,
    filter="icmp",
    lfilter=skip_loopback_copies(),
    store=0,
)

### Task 1 – Receiver for UDP, one character per packet

Adapt the receiver: a capture filter for UDP to `DST_PORT` (`f"udp and dst port {DST_PORT}"`) and a callback that prints the one-character payload of each packet without a line break (`end="", flush=True`). Keep `lfilter=skip_loopback_copies()` when testing on `lo`.

### Task 2 – Random delay and loss

Keep the Task 1 receiver running while the sender drops and delays packets. Which characters are missing, and does the receiver notice that anything is missing? What would it need to notice?

### 2.2 Audio over RTP

Captures for `CAPTURE_SECONDS`; start the cell, then the sender in `lab_03.ipynb`. The capture is saved to `results/received.pcap`, analyzed, and decoded back into audio. Missing packets leave silence.

In [ ]:
print(
    f"Capturing {CAPTURE_SECONDS} s on {IFACE}, UDP port {DST_PORT}: start the sender now."
)
captured = sa.sniff(
    iface=IFACE,
    filter=f"udp and dst port {DST_PORT}",
    lfilter=skip_loopback_copies(),
    timeout=CAPTURE_SECONDS,
)
sa.wrpcap(str(RESULTS_DIR / "received.pcap"), captured)

received = rtp_stream(captured, dport=DST_PORT)
stream_view(received, stream_stats(received), title="Received stream")

In [ ]:
Audio(decode_rtp(received, ENCODING) / 32768, rate=SAMPLE_RATE, normalize=False)

## Step 3 – Emulate impairments with `tc`

The receiver's capture of each trial is the authoritative one. Start `receive_trial` with the trial name, then `run_trial` with the same name in `lab_03.ipynb`. The capture lasts a few seconds longer than the test stream and is saved to `results/received_<name>.pcap`. Because the receiver does not know the first sequence number of the stream, losses before the first received packet are not counted.

In [ ]:
trials = {}


def receive_trial(name, R=None, margin=10):
    # Capture one trial, save it, and analyze it.
    print(
        f"Capturing trial {name!r} for {TRIAL_SECONDS + margin} s: start run_trial now."
    )
    captured = sa.sniff(
        iface=IFACE,
        filter=f"udp and dst port {DST_PORT}",
        lfilter=skip_loopback_copies(),
        timeout=TRIAL_SECONDS + margin,
    )
    sa.wrpcap(str(RESULTS_DIR / f"received_{name}.pcap"), captured)
    stream = rtp_stream(captured, dport=DST_PORT)
    trials[name] = stats = stream_stats(stream, expected=TRIAL_PACKETS)
    return stream_view(stream, stats, R=R, title=f"Trial {name}, receiver")

In [ ]:
receive_trial("baseline")

## Step 4 – Replayed call

Captures one replay of the recorded call (`replay_rtp` in `lab_03.ipynb`, README Step 4) and saves it as `results/output_<model>_<loss>.pcap`, next to the WAV file of the same name that README Step 5 reconstructs from it; the quality evaluation in `lab_03.ipynb` then reads the measured loss and jitter from it. Start the cell first, then the replay. Raise `seconds` above the length of the call.

In [ ]:
def receive_replay(model, loss, seconds=30):
    # Capture one replay, save it under the name of its condition, and analyze it.
    path = RESULTS_DIR / f"output_{model}_{loss}.pcap"
    print(
        f"Capturing {seconds} s on {IFACE}, UDP port {DST_PORT}: start the replay now."
    )
    captured = sa.sniff(
        iface=IFACE,
        filter=f"udp and dst port {DST_PORT}",
        lfilter=skip_loopback_copies(),
        timeout=seconds,
    )
    sa.wrpcap(str(path), captured)
    stream = rtp_stream(captured, dport=DST_PORT)
    display(stream_card(stream_stats(stream), path.name))
    return path

In [ ]:
receive_replay("random", "0.5")